# XAI Traffic Forecasting — Batch 4 (Colab Pro / GPU)
**Scope:** SEQ_LEN ablation di granularity 10-menit (144 vs 288 langkah) + multivariate input (n_flows, n_packets, tcp_udp_ratio, avg_ttl).

Kenapa di Colab: training granularity 10-menit di laptop (CPU) makan ~50 menit/seed untuk SEQ_LEN=144 (28.071 sequence). SEQ_LEN=288 dan multivariate nambah beban lebih jauh — GPU T4/A100 di Colab Pro jauh lebih cepat buat ini.

**Cara pakai:**
1. Upload notebook ini ke Colab (Runtime > Change runtime type > GPU)
2. Jalankan cell "Setup" — upload 2 file dataset yang diminta (lihat instruksi di cell itu)
3. Jalankan semua cell berurutan
4. Download hasil (`results_batch4.json`) di akhir, kirim balik ke Claude Code buat digabung ke `DATASET_DECISION_LOG.md`

## 0. Setup — Install & Upload Data

In [8]:
!pip install -q tensorflow keras pandas numpy scikit-learn

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import time, json

print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

GPU tersedia: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


### Upload dataset

Butuh 2 file dari laptop (folder `datasets/cesnet/` di project):
1. `global_bytes_10min.csv` — target n_bytes
2. Kalau mau eksperimen multivariate juga: file per-institution dari `institutions/agg_10_minutes/*.csv` sudah diagregasi jadi kolom tambahan (`n_flows`, `n_packets`, `tcp_udp_ratio_bytes`, `avg_ttl`) — siapkan file gabungan ini dulu di laptop sebelum upload (lihat catatan di akhir notebook kalau belum ada agregasinya).

Jalankan cell di bawah, klik "Choose Files", pilih file-nya.

In [9]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Load & Preprocessing (SEQ_LEN=144, baseline pembanding)

In [10]:
DATASET_PATH = "/content/drive/MyDrive/farrel/xai/global_bytes_10min.csv"  # sesuaikan nama file kalau beda
TIMES_PATH = "/content/drive/MyDrive/farrel/xai/times_10_minutes.csv"       # upload juga file ini kalau belum

bytes_df = pd.read_csv(DATASET_PATH)
times_df = pd.read_csv(TIMES_PATH)
df_raw = times_df.merge(bytes_df, on="id_time", how="left")
df_raw["datetime"] = pd.to_datetime(df_raw["time"], utc=True)
df_raw = df_raw.set_index("datetime").sort_index()

series = (
    df_raw["n_bytes"]
    .resample("10min", closed="left", label="left")
    .sum()
    .replace(0, np.nan)
    .ffill()
    .fillna(0)
)
values = series.values.reshape(-1, 1)
print("Series length:", len(values))

Series length: 40308


## 2. Arsitektur (sama seperti model utama, custom AttentionContext layer)

**PENTING:** class ini HARUS didefinisikan sebelum load/build model apapun di Colab (custom Keras layer perlu terdaftar di proses yang aktif).

In [11]:
@keras.saving.register_keras_serializable(package="xai_traffic_colab")
class AttentionContext(layers.Layer):
    def call(self, inputs):
        hidden_states, attn_weights = inputs
        return tf.reduce_sum(hidden_states * attn_weights, axis=1)

    def compute_output_shape(self, input_shape):
        hidden_shape, _ = input_shape
        return (hidden_shape[0], hidden_shape[2])


def build_model(seq_len, n_features=1, units=96, dropout=0.0):
    inputs = keras.Input(shape=(seq_len, n_features), name="input_sequence")
    lstm_out = layers.LSTM(units, return_sequences=True, dropout=dropout, name="lstm")(inputs)
    score = layers.Dense(1, activation="tanh", name="attention_score")(lstm_out)
    attn_weights = layers.Softmax(axis=1, name="attention_weights")(score)
    context = AttentionContext(name="context_vector")([lstm_out, attn_weights])
    output = layers.Dense(1, name="volume_output")(context)
    return keras.Model(inputs=inputs, outputs=output, name="lstm_attention_colab")


def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i + seq_len])
        y.append(data[i + seq_len])
    return np.array(X), np.array(y)


def run_experiment(values, seq_len, n_features=1, label=""):
    n = len(values)
    n_train, n_val = int(n * 0.70), int(n * 0.15)
    train_raw, val_raw, test_raw = values[:n_train], values[n_train:n_train+n_val], values[n_train+n_val:]

    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(train_raw[:, :1] if train_raw.ndim > 1 else train_raw.reshape(-1,1))
    # scaling: kalau multivariate, scale kolom target (index 0) saja, biarkan fitur lain (rasio) di range asli
    def scale_target_only(arr):
        out = arr.copy().astype(float)
        out[:, 0:1] = scaler.transform(arr[:, 0:1])
        return out

    if n_features > 1:
        train_s, val_s, test_s = scale_target_only(train_raw), scale_target_only(val_raw), scale_target_only(test_raw)
    else:
        train_s = scaler.transform(train_raw)
        val_s = scaler.transform(val_raw)
        test_s = scaler.transform(test_raw)

    X_train, y_train = create_sequences(train_s, seq_len)
    X_val, y_val = create_sequences(val_s, seq_len)
    X_test, y_test = create_sequences(test_s, seq_len)
    if n_features > 1:
        y_train, y_val, y_test = y_train[:, 0], y_val[:, 0], y_test[:, 0]

    tf.random.set_seed(42); np.random.seed(42)
    model = build_model(seq_len, n_features=n_features)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]

    t0 = time.time()
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100,
                      batch_size=32, callbacks=cb, verbose=1)
    elapsed = time.time() - t0

    y_pred_scaled = model.predict(X_test, verbose=0)
    y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true_col = y_test[:, 0] if y_test.ndim > 1 else y_test
    y_true = scaler.inverse_transform(y_true_col.reshape(-1, 1)).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    n_epochs = len(hist.history["loss"])
    print(f"[{label}] SEQ_LEN={seq_len} n_features={n_features} -> RMSE={rmse:.4e} MAE={mae:.4e} "
          f"epochs={n_epochs} time={elapsed:.1f}s")

    return {"label": label, "seq_len": seq_len, "n_features": n_features, "rmse": float(rmse),
            "mae": float(mae), "epochs": len(hist.history["loss"]), "time_s": round(elapsed, 1)}

## 3. Jalankan SEQ_LEN Ablation (144 vs 288)

In [12]:
results = []
results.append(run_experiment(values, seq_len=144, label="seqlen_144_baseline"))
results.append(run_experiment(values, seq_len=288, label="seqlen_288"))

for r in results:
    print(r)

Epoch 1/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.0083 - mae: 0.0704 - val_loss: 0.0052 - val_mae: 0.0544
Epoch 2/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - loss: 0.0049 - mae: 0.0532 - val_loss: 0.0045 - val_mae: 0.0497
Epoch 3/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0040 - mae: 0.0473 - val_loss: 0.0028 - val_mae: 0.0383
Epoch 4/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0030 - mae: 0.0404 - val_loss: 0.0024 - val_mae: 0.0335
Epoch 5/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0026 - mae: 0.0378 - val_loss: 0.0022 - val_mae: 0.0324
Epoch 6/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0025 - mae: 0.0367 - val_loss: 0.0022 - val_mae: 0.0325
Epoch 7/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0024 - mae: 0.0360 - val_loss: 0.0021 - val_mae: 0.0315
Epoch 8/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0023 - mae: 0.0353 - val_loss: 0.0020 - val_mae: 0.0312
Epoch 9/100
878/878 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms

## 4. (Opsional) Multivariate Input

Kalau sudah upload file dengan kolom tambahan (`n_flows`, `n_packets`, `tcp_udp_ratio_bytes`, `avg_ttl`), jalankan cell ini.
Kalau belum, skip ke bagian 5.

In [13]:
# Ganti path sesuai file multivariate yang di-upload
MULTIVARIATE_PATH = "global_multivariate_10min.csv"  # placeholder, sesuaikan

try:
    multi_df = pd.read_csv(MULTIVARIATE_PATH)
    feature_cols = ["n_bytes", "n_flows", "n_packets", "tcp_udp_ratio_bytes", "avg_ttl"]
    multi_values = multi_df[feature_cols].values
    results.append(run_experiment(multi_values, seq_len=144, n_features=len(feature_cols), label="multivariate_seqlen144"))
    print(results[-1])
except FileNotFoundError:
    print("File multivariate belum di-upload, skip bagian ini.")

File multivariate belum di-upload, skip bagian ini.


## 5. Simpan & Download Hasil

In [14]:
with open("results_batch4.json", "w") as f:
    json.dump(results, f, indent=2)

from google.colab import files
files.download("results_batch4.json")
print("Hasil tersimpan & siap di-download: results_batch4.json")
print("Kirim file ini balik ke sesi Claude Code buat digabung ke DATASET_DECISION_LOG.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Hasil tersimpan & siap di-download: results_batch4.json
Kirim file ini balik ke sesi Claude Code buat digabung ke DATASET_DECISION_LOG.md
